# Gold — Race Summary Facts
Uma linha por corrida com os principais fatos agregados: vencedor, volta mais rápida, estatísticas gerais e pontuação distribuída.

In [ ]:
import dlt
from pyspark.sql.functions import (
    col, sum, count, min, max, avg, when, first,
    round as spark_round
)
from pyspark.sql.window import Window

spark.sql("USE CATALOG f1_lakehouse")

In [ ]:
@dlt.table(
    name="race_summary_facts",
    comment="Fatos agregados por corrida: vencedor, volta mais rápida e estatísticas gerais",
    table_properties={"quality": "gold"},
    partition_cols=["season"]
)
@dlt.expect_all({
    "valid_season":    "season IS NOT NULL",
    "valid_round":     "round IS NOT NULL",
    "has_starters":    "total_starters > 0"
})
def race_summary_facts():
    races_df   = dlt.read("races")
    results_df = dlt.read("results")

    # -------------------------------------------------------
    # 1. Estatísticas gerais por corrida
    # -------------------------------------------------------
    stats = (
        results_df
        .groupBy("season", col("race_round").cast("int").alias("round"))
        .agg(
            count("driver_id").alias("total_starters"),
            sum(
                when(col("final_position").isNotNull(), 1).otherwise(0)
            ).alias("total_classified"),
            sum(
                when(col("position_text").isin("R", "D", "E", "W", "F", "N"), 1).otherwise(0)
            ).alias("dnf_count"),
            sum("points").alias("total_points_awarded"),
            spark_round(avg("grid_position"), 2).alias("avg_grid_position"),
            max("laps").alias("race_laps")           # laps do vencedor (max = quem completou mais)
        )
    )

    # -------------------------------------------------------
    # 2. Vencedor (final_position == 1)
    # -------------------------------------------------------
    winner = (
        results_df
        .filter(col("final_position") == 1)
        .select(
            col("season"),
            col("race_round").cast("int").alias("round"),
            col("driver_id").alias("winner_driver_id"),
            col("driver_name").alias("winner_driver_name"),
            col("constructor_id").alias("winner_constructor_id"),
            col("finish_time").alias("winning_time"),
            col("milliseconds").alias("winning_milliseconds")
        )
    )

    # -------------------------------------------------------
    # 3. Segundo colocado — para calcular margem de vitória
    # -------------------------------------------------------
    second = (
        results_df
        .filter(col("final_position") == 2)
        .select(
            col("season"),
            col("race_round").cast("int").alias("round"),
            col("milliseconds").alias("second_milliseconds")
        )
    )

    # -------------------------------------------------------
    # 4. Volta mais rápida (fastest_lap rank == 1)
    # -------------------------------------------------------
    fastest = (
        results_df
        .filter(col("fastest_lap") == 1)
        .select(
            col("season"),
            col("race_round").cast("int").alias("round"),
            col("driver_id").alias("fastest_lap_driver_id"),
            col("driver_name").alias("fastest_lap_driver_name"),
            col("fastest_lap_time"),
            col("fastest_lap_speed")
        )
    )

    # -------------------------------------------------------
    # 5. Pole position (grid_position == 1)
    # -------------------------------------------------------
    pole = (
        results_df
        .filter(col("grid_position") == 1)
        .select(
            col("season"),
            col("race_round").cast("int").alias("round"),
            col("driver_id").alias("pole_driver_id"),
            col("driver_name").alias("pole_driver_name")
        )
    )

    # -------------------------------------------------------
    # 6. Joins
    # -------------------------------------------------------
    race_keys = ["season", "round"]

    df = (
        races_df
        .select(
            col("season"),
            col("round"),
            col("race_name"),
            col("race_date"),
            col("circuit_id")
        )
        .join(stats,   on=race_keys, how="left")
        .join(winner,  on=race_keys, how="left")
        .join(second,  on=race_keys, how="left")
        .join(fastest, on=race_keys, how="left")
        .join(pole,    on=race_keys, how="left")
    )

    # Margem de vitória em ms (null quando sem dado de tempo)
    df = df.withColumn(
        "winning_margin_ms",
        when(
            col("winning_milliseconds").isNotNull() & col("second_milliseconds").isNotNull(),
            col("second_milliseconds") - col("winning_milliseconds")
        ).otherwise(None)
    ).drop("second_milliseconds")

    return (
        df
        .select(
            # Identificação da corrida
            "season",
            "round",
            "race_name",
            "race_date",
            "circuit_id",
            # Vencedor
            "winner_driver_id",
            "winner_driver_name",
            "winner_constructor_id",
            "winning_time",
            "winning_milliseconds",
            "winning_margin_ms",
            # Pole position
            "pole_driver_id",
            "pole_driver_name",
            # Volta mais rápida
            "fastest_lap_driver_id",
            "fastest_lap_driver_name",
            "fastest_lap_time",
            "fastest_lap_speed",
            # Estatísticas gerais
            "race_laps",
            "total_starters",
            "total_classified",
            "dnf_count",
            "total_points_awarded",
            "avg_grid_position"
        )
        .orderBy("season", "round")
    )